# ForgeGuard: Full-Dataset Training Pipeline (Google Colab)
### A Cross-Architecture Analysis of CNN Models in Detecting Forged Digital Transaction Receipts
**Notre Dame of Midsayap College (NDMC) — BSCS Thesis Writing 1**  
**Researchers**: Rogie P. Bacanto, Daniela S. Ungab | **Adviser**: Ms. Doris Ann Mariano  

---
### Experimental Full-Dataset Protocol
This notebook trains and evaluates the three CNN architectures using **ALL available dataset samples** (228 Authentic + 849 Forged across all subcategories = 1,077 total receipts):
1. **Basic CNN**: Custom 3-layer sequential network (~2.1M params)
2. **MobileNetV2**: Lightweight inverted residual depthwise separable CNN (~3.4M params)
3. **ResNet50**: 50-layer deep residual network (~23.5M params)

**Balance Protection**: Because the dataset contains 849 forged receipts vs. 228 authentic receipts (~80% fake), this notebook uses **Balanced Class Weights** (`compute_class_weight`) to penalize false alarms and ensure the models do not become biased toward flagging genuine receipts.

**Isolated Storage**: All newly trained models and metrics are saved to `models_full_dataset/` to protect your official thesis proposal benchmarks.

In [ ]:
# Step 1: Clone repository & install dependencies
!git clone https://github.com/DeathKnell837/ForgeGuard.git
%cd ForgeGuard/thesis-system
!pip install -q Pillow numpy scipy scikit-learn matplotlib seaborn

In [ ]:
# Step 2: Verify GPU acceleration
import os, glob, time, json, shutil
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU Accelerated:", gpus[0])
else:
    print("WARNING: Running on CPU. In Google Colab, go to Runtime -> Change runtime type -> T4 GPU for faster training.")

In [ ]:
# Step 3: Load ALL Authentic & ALL Forged Receipts (No Cutoffs)
from preprocessing.ela import compute_ela

IMG_SIZE = (128, 128)
IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.webp")

auth_dir = "dataset/authentic/compressed"
forged_dir = "dataset/forged/compressed"

# 1. Load all Authentic Receipts (Label: 0)
auth_files = []
for ext in IMAGE_EXTENSIONS:
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext)))
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext.upper())))
auth_files = sorted(list(set(auth_files)))
print(f"[1/3] Found {len(auth_files)} Authentic receipts.")

# 2. Load ALL Forged Receipts across all categories (Label: 1)
all_forged_files = []
subcategories = [
    "amount_alteration",
    "name_modification",
    "ref_fabrication",
    "font_tampering",
    "ai_generated_template",
    "ai_diffusion_generated",
    "full_template"
]

for subcat in subcategories:
    sub_files = []
    for ext in IMAGE_EXTENSIONS:
        sub_files.extend(glob.glob(os.path.join(forged_dir, subcat, ext)))
        sub_files.extend(glob.glob(os.path.join(forged_dir, subcat, ext.upper())))
    sub_files = sorted(list(set(sub_files)))
    all_forged_files.extend(sub_files)
    print(f"      - {subcat:25s}: {len(sub_files)} samples")

all_forged_files = sorted(list(set(all_forged_files)))
print(f"[2/3] Total Forged Receipts: {len(all_forged_files)}")
print(f"[3/3] Total Dataset: {len(auth_files) + len(all_forged_files)} receipts")

# Preprocess Full Dataset using ELA
X, y = [], []
print("\nExtracting ELA features for Authentic receipts...")
for f in auth_files:
    with Image.open(f) as img:
        ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
        X.append(np.array(ela, dtype=np.float32) / 255.0)
        y.append(0)

print("Extracting ELA features for ALL Forged receipts...")
for f in all_forged_files:
    with Image.open(f) as img:
        ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
        X.append(np.array(ela, dtype=np.float32) / 255.0)
        y.append(1)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

# Stratified Train (70%) / Val (15%) / Test (15%) Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Calculate Balanced Class Weights
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print(f"\nTrain: {len(X_train)} (Auth:{np.sum(y_train==0)}, Fake:{np.sum(y_train==1)})")
print(f"Val:   {len(X_val)} (Auth:{np.sum(y_val==0)}, Fake:{np.sum(y_val==1)})")
print(f"Test:  {len(X_test)} (Auth:{np.sum(y_test==0)}, Fake:{np.sum(y_test==1)})")
print(f"Class Weights: Authentic={class_weight_dict[0]:.2f}, Forged={class_weight_dict[1]:.2f}")

In [ ]:
# Step 4: Define Model Architectures
def build_basic_cnn():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_mobilenetv2():
    base = applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights="imagenet")
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_resnet50():
    base = applications.ResNet50(input_shape=(128, 128, 3), include_top=False, weights="imagenet")
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss="binary_crossentropy", metrics=["accuracy"])
    return model

print("Model architectures initialized.")

In [ ]:
# Step 5: Train Models with Class Weights & Save to models_full_dataset/
save_dir = "models_full_dataset"
os.makedirs(save_dir, exist_ok=True)

models_dict = {
    "Basic_CNN": build_basic_cnn(),
    "MobileNetV2": build_mobilenetv2(),
    "ResNet50": build_resnet50()
}

full_results = {}

for name, model in models_dict.items():
    print(f"\n{'='*25} Training {name} on Full Dataset {'='*25}")
    t0 = time.time()
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=16,
        class_weight=class_weight_dict,
        verbose=1
    )
    train_time = time.time() - t0
    
    # Evaluate on Test Split
    t_inf = time.time()
    y_prob = model.predict(X_test, verbose=0)
    lat_ms = ((time.time() - t_inf) / len(X_test)) * 1000.0
    y_pred = (y_prob >= 0.5).astype(int).flatten()
    
    acc = float(accuracy_score(y_test, y_pred))
    prec = float(precision_score(y_test, y_pred, zero_division=0))
    rec = float(recall_score(y_test, y_pred, zero_division=0))
    f1 = float(f1_score(y_test, y_pred, zero_division=0))
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    full_results[name] = {
        "architecture": name.replace("_", " "),
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "latency_ms": float(lat_ms),
        "train_time_s": float(train_time),
        "confusion": {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    }
    
    out_model = f"{save_dir}/{name.lower()}_full.keras"
    model.save(out_model)
    print(f"\nSaved: {out_model}")
    print(f"Test Acc: {acc*100:.2f}% | Prec: {prec*100:.2f}% | Rec: {rec*100:.2f}% | F1: {f1:.4f} | Lat: {lat_ms:.2f}ms")

# Save separate metrics file (Official benchmarks untouched)
with open(f"{save_dir}/full_dataset_metrics.json", "w") as f:
    json.dump(full_results, f, indent=2)
print(f"\nFull dataset metrics saved to {save_dir}/full_dataset_metrics.json")

In [ ]:
# Step 6: Print Summary & Download ZIP
print("="*75)
print("FORGEGUARD: FULL DATASET EVALUATION SUMMARY")
print("="*75)
print(f"{'Architecture':15s} | {'Accuracy':9s} | {'Precision':9s} | {'Recall':9s} | {'F1-Score':9s} | {'Latency':10s}")
print("-"*75)
for name, v in full_results.items():
    print(f"{v['architecture']:15s} | {v['accuracy']*100:6.2f}%   | {v['precision']*100:6.2f}%   | {v['recall']*100:6.2f}%   | {v['f1_score']:6.4f}    | {v['latency_ms']:6.2f} ms")
print("="*75)

zip_name = "forgeguard_full_dataset_models"
shutil.make_archive(zip_name, "zip", save_dir)
print(f"\nArchive created: {zip_name}.zip")

try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
    print("Download initiated automatically in Colab!")
except Exception:
    print(f"Saved locally at: {zip_name}.zip")